# Модели: логистическая регрессия и Random Forest

Приоритет: **не пропускать нарушения** → смотрим **Recall класса 0**. Используем `class_weight='balanced'`.

**Зависимость:** сначала выполните `03_preprocessing.ipynb` (файл `data/processed/ml_ready.joblib`).

**Colab:** [colab_quickstart.ipynb](colab_quickstart.ipynb); sklearn **не использует GPU**.

In [1]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

def _project_root() -> Path:
    spec = importlib.util.find_spec('data_loader')
    if spec is not None and getattr(spec, 'origin', None):
        cand = Path(spec.origin).resolve().parent.parent
        if (cand / 'data' / 'raw').is_dir() or (cand / 'src' / 'data_loader.py').is_file():
            return cand
    env = os.environ.get('WATER_QUALITY_EE_ROOT', '').strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / 'src' / 'data_loader.py').is_file():
            return p
    cwd = Path.cwd().resolve()
    try:
        r = subprocess.run(
            ['git', 'rev-parse', '--show-toplevel'],
            cwd=cwd, capture_output=True, text=True, timeout=15,
        )
        if r.returncode == 0 and r.stdout.strip():
            p = Path(r.stdout.strip()).resolve()
            if (p / 'src' / 'data_loader.py').is_file():
                return p
    except (FileNotFoundError, OSError, subprocess.TimeoutExpired):
        pass
    for root in [cwd, *list(cwd.parents)[:28]]:
        if (root / 'src' / 'data_loader.py').is_file():
            return root
    raise RuntimeError('См. 01: pip install -e . или корректный WATER_QUALITY_EE_ROOT.')

if importlib.util.find_spec('data_loader') is None:
    sys.path.insert(0, str(_project_root() / 'src'))

import data_loader
ROOT = Path(data_loader.__file__).resolve().parent.parent

import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from evaluate import evaluate_model, compare_models

In [2]:
path = ROOT / 'data' / 'processed' / 'ml_ready.joblib'
if not path.exists():
    raise FileNotFoundError(f'Нет {path}. Запустите notebooks/03_preprocessing.ipynb')
b = joblib.load(path)
X_train, X_test = b['X_train'], b['X_test']
y_train, y_test = b['y_train'], b['y_test']
feature_names = b['feature_names']

## 1. Logistic Regression

In [3]:
lr = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
res_lr = evaluate_model(lr, X_test, y_test, model_name='Logistic Regression')


Модель: Logistic Regression
               precision    recall  f1-score   support

Нарушение (0)       0.44      0.81      0.57       548
    Норма (1)       0.98      0.92      0.95      7171

     accuracy                           0.91      7719
    macro avg       0.71      0.87      0.76      7719
 weighted avg       0.95      0.91      0.92      7719

ROC-AUC: 0.9241


## 2. Random Forest

In [4]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
res_rf = evaluate_model(rf, X_test, y_test, model_name='Random Forest')


Модель: Random Forest
               precision    recall  f1-score   support

Нарушение (0)       0.88      0.91      0.89       548
    Норма (1)       0.99      0.99      0.99      7171

     accuracy                           0.98      7719
    macro avg       0.94      0.95      0.94      7719
 weighted avg       0.98      0.98      0.98      7719

ROC-AUC: 0.9852


## 3. Сводная таблица

In [5]:
compare_models([res_lr, res_rf])

,Accuracy,Precision (нарушение),Recall (нарушение),F1 (нарушение),ROC-AUC
Модель,,,,,
Logistic Regression,0.9127,0.4380,0.8120,0.5691,0.9241
Random Forest,0.9846,0.8796,0.9069,0.8931,0.9852


## 4. Сохранение моделей для ноутбука 05

In [6]:
out = ROOT / 'data' / 'processed' / 'trained_models.joblib'
joblib.dump({'lr': lr, 'rf': rf, 'results': [res_lr, res_rf]}, out)
print('Сохранено:', out)

Сохранено: /home/anton/projects/water-quality-ee/data/processed/trained_models.joblib
